# Notebook 02 — Treinamento e Profiling de Modelos

## Aula 01: Fundamentos de Latência e Performance em ML

---

### 🎯 Objetivos deste Notebook

1. Treinar múltiplos algoritmos de classificação (LR, RF, GB, SVM)
2. Medir e comparar latência de treinamento e inferência
3. Analisar o impacto do batch size na relação latência-throughput
4. Realizar hyperparameter tuning com medição de custo computacional
5. Realizar cross-validation com profiling de tempo

### 📖 Conceito Teórico

Conforme a seção 'Complexidade de Modelos', a complexidade de inferência varia entre algoritmos:

| Algoritmo | Complexidade de Inferência | Observação |
|-----------|---------------------------|------------|
| Regressão Logística | $O(d)$ | Linear no nº de features |
| Random Forest | $O(t \cdot \log n)$ | t = nº árvores |
| Gradient Boosting | $O(t \cdot \text{profundidade})$ | Sequencial por natureza |
| SVM (kernel RBF) | $O(n_{sv} \cdot d)$ | $n_{sv}$ = support vectors |

Referência: Reddi, V. J. et al. (2020). *MLPerf Inference Benchmark*. ISCA '20.

---

In [ ]:
# Imports necessários
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Adicionar diretório raiz ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.model import LatencyBenchmarkModel
from src.training import (
    cross_validate_model,
    hyperparameter_tuning,
    train_model,
    train_multiple_models,
)
from src.utils import save_model, save_metrics, setup_output_dirs

# Configurações
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)

dirs = setup_output_dirs(str(Path.cwd().parent))
print("Ambiente configurado com sucesso!")

## 1. Carregar Dados Processados

Os dados foram processados no Notebook 01 e salvos como arrays NumPy.

In [ ]:
# Carregar splits processados no Notebook 01
processed_dir = Path.cwd().parent / "data" / "processed"

X_train = np.load(str(processed_dir / "X_train.npy"))
X_val = np.load(str(processed_dir / "X_val.npy"))
X_test = np.load(str(processed_dir / "X_test.npy"))
y_train = np.load(str(processed_dir / "y_train.npy"))
y_val = np.load(str(processed_dir / "y_val.npy"))
y_test = np.load(str(processed_dir / "y_test.npy"))

print("Dados carregados:")
print(f"  X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}    |  y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}  |  y_test:  {y_test.shape}")

## 2. Treinamento de Múltiplos Modelos

Vamos treinar 4 algoritmos e comparar a **latência de treinamento**.
Embora a latência de treinamento não afete diretamente a experiência
do usuário final, ela impacta o ciclo de retraining em produção
(Sculley et al., 2015).

In [ ]:
# Treinar todos os modelos suportados
results = train_multiple_models(X_train, y_train, random_state=42)

# Tabela de resultados de treinamento
training_metrics = [m for _, m in results]
training_df = pd.DataFrame(training_metrics)

print("=" * 70)
print("LATÊNCIA DE TREINAMENTO POR MODELO")
print("=" * 70)
print(training_df.to_string(index=False))

In [ ]:
# Visualização da latência de treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_names = training_df["model_name"]
x = np.arange(len(models_names))

# Gráfico 1: Tempo de treinamento
bars = axes[0].bar(x, training_df["training_time_s"], color="steelblue", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_names, rotation=45, ha="right")
axes[0].set_ylabel("Tempo de Treinamento (s)", fontsize=12)
axes[0].set_title("Latência de Treinamento por Modelo", fontsize=13)
axes[0].grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, training_df["training_time_s"]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f"{val:.3f}s", ha="center", va="bottom", fontsize=10)

# Gráfico 2: Samples/segundo durante treinamento
bars2 = axes[1].bar(x, training_df["samples_per_second"], color="seagreen", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models_names, rotation=45, ha="right")
axes[1].set_ylabel("Amostras/segundo", fontsize=12)
axes[1].set_title("Throughput de Treinamento", fontsize=13)
axes[1].grid(True, alpha=0.3, axis="y")

fig.suptitle("Comparação de Performance de Treinamento", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(str(dirs["figures"] / "02_training_latency.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Benchmark de Latência de Inferência

A **latência de inferência** ($L_{inferência}$) é a métrica mais crítica
em produção. Conforme o protocolo MLPerf (Reddi et al., 2020), medimos
com múltiplas iterações após warm-up para capturar a distribuição real.

Calculamos os percentis P50, P95 e P99, que são os KPIs mais relevantes
para SLAs em serving systems (Dean & Barroso, 2013).

In [ ]:
# Benchmark de inferência para todos os modelos
models = [m for m, _ in results]
benchmark_results = []

for model in models:
    print(f"Benchmarking {model.model_name}...")
    bench = model.benchmark_inference(X_test, n_iterations=200)
    benchmark_results.append(bench)

benchmark_df = pd.DataFrame(benchmark_results)

print("\n" + "=" * 80)
print("BENCHMARK DE LATÊNCIA DE INFERÊNCIA (ms)")
print("=" * 80)
display_cols = ["model", "n_samples", "mean_ms", "std_ms", "p50_ms", "p95_ms", "p99_ms", "throughput_samples_per_sec"]
print(benchmark_df[display_cols].to_string(index=False))

In [ ]:
# Visualização completa dos benchmarks
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_labels = benchmark_df["model"].values
x = np.arange(len(model_labels))

# 1. Latência média ± desvio
axes[0].bar(x, benchmark_df["mean_ms"], yerr=benchmark_df["std_ms"],
            capsize=5, color="steelblue", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_labels, rotation=45, ha="right")
axes[0].set_ylabel("Latência (ms)")
axes[0].set_title("Latência Média de Inferência")
axes[0].grid(True, alpha=0.3, axis="y")

# 2. Percentis P50/P95/P99
width = 0.25
axes[1].bar(x - width, benchmark_df["p50_ms"], width, label="P50", alpha=0.8)
axes[1].bar(x, benchmark_df["p95_ms"], width, label="P95", alpha=0.8)
axes[1].bar(x + width, benchmark_df["p99_ms"], width, label="P99", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_labels, rotation=45, ha="right")
axes[1].set_ylabel("Latência (ms)")
axes[1].set_title("Percentis de Latência (P50/P95/P99)")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

# 3. Throughput
axes[2].barh(x, benchmark_df["throughput_samples_per_sec"], color="seagreen", alpha=0.8)
axes[2].set_yticks(x)
axes[2].set_yticklabels(model_labels)
axes[2].set_xlabel("Amostras/segundo")
axes[2].set_title("Throughput de Inferência")
axes[2].grid(True, alpha=0.3, axis="x")

fig.suptitle("Comparação de Performance de Inferência entre Modelos", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(str(dirs["figures"] / "02_inference_benchmark.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Análise de Tail Latency — Predição Individual

Conforme Dean & Barroso (2013), *"The Tail at Scale"*, a **tail latency**
é particularmente problemática em sistemas distribuídos:

> Se um serviço distribui um request para 100 backend servers em paralelo,
> e cada server tem 1% de chance de uma latência > 100ms, então o serviço
> terá ~63% de chance de experimentar essa latência em cada request.

Vamos medir a distribuição de latência para predições individuais.

In [ ]:
# Medir latência de predição individual (1000 amostras)
n_single = 1000
single_latencies = {}

for model in models:
    latencies = []
    for i in range(n_single):
        _, lat_ms = model.predict_single(X_test[i % len(X_test)])
        latencies.append(lat_ms)
    single_latencies[model.model_name] = latencies

# Visualizar distribuição
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, lats) in enumerate(single_latencies.items()):
    arr = np.array(lats)
    axes[idx].hist(arr, bins=50, color="steelblue", alpha=0.7, edgecolor="white")
    axes[idx].axvline(np.percentile(arr, 50), color="green", linestyle="--",
                      label=f"P50: {np.percentile(arr, 50):.3f} ms")
    axes[idx].axvline(np.percentile(arr, 95), color="orange", linestyle="--",
                      label=f"P95: {np.percentile(arr, 95):.3f} ms")
    axes[idx].axvline(np.percentile(arr, 99), color="red", linestyle="--",
                      label=f"P99: {np.percentile(arr, 99):.3f} ms")
    axes[idx].set_title(f"{name}", fontsize=12)
    axes[idx].set_xlabel("Latência (ms)")
    axes[idx].set_ylabel("Frequência")
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

fig.suptitle("Distribuição de Latência — Predição Individual (n=1000)", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(str(dirs["figures"] / "02_tail_latency_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Impacto do Batch Size na Latência e Throughput

O **batch size** é um dos parâmetros mais importantes para otimizar
o trade-off entre latência e throughput:

- **Batch size pequeno**: Menor latência individual, menor throughput
- **Batch size grande**: Maior latência individual, maior throughput

Conforme a seção 'Throughput vs. Latência', o ponto ótimo depende
dos SLAs da aplicação.

In [ ]:
from src.evaluation import benchmark_batch_sizes

# Analisar batch sizes para o modelo Random Forest
rf_model = models[1]  # random_forest
batch_sizes = [1, 5, 10, 50, 100, 200, 500, 1000]

batch_df = benchmark_batch_sizes(
    rf_model, X_test, batch_sizes=batch_sizes, n_iterations=100
)

print("=" * 70)
print(f"ANÁLISE DE BATCH SIZE — {rf_model.model_name}")
print("=" * 70)
print(batch_df.to_string(index=False))

In [ ]:
# Visualização do trade-off batch size
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Latência total vs batch size
axes[0].plot(batch_df["batch_size"], batch_df["mean_latency_ms"], "o-",
             color="steelblue", linewidth=2)
axes[0].fill_between(batch_df["batch_size"], batch_df["p50_ms"],
                     batch_df["p99_ms"], alpha=0.2, color="steelblue")
axes[0].set_xlabel("Batch Size", fontsize=12)
axes[0].set_ylabel("Latência Total (ms)", fontsize=12)
axes[0].set_title("Latência vs. Batch Size", fontsize=13)
axes[0].set_xscale("log")
axes[0].grid(True, alpha=0.3)

# 2. Latência por amostra vs batch size
axes[1].plot(batch_df["batch_size"], batch_df["latency_per_sample_ms"], "s-",
             color="coral", linewidth=2)
axes[1].set_xlabel("Batch Size", fontsize=12)
axes[1].set_ylabel("Latência por Amostra (ms)", fontsize=12)
axes[1].set_title("Latência Unitária vs. Batch Size", fontsize=13)
axes[1].set_xscale("log")
axes[1].grid(True, alpha=0.3)

# 3. Throughput vs batch size
axes[2].plot(batch_df["batch_size"], batch_df["throughput_samples_per_sec"], "^-",
             color="seagreen", linewidth=2)
axes[2].set_xlabel("Batch Size", fontsize=12)
axes[2].set_ylabel("Throughput (amostras/s)", fontsize=12)
axes[2].set_title("Throughput vs. Batch Size", fontsize=13)
axes[2].set_xscale("log")
axes[2].grid(True, alpha=0.3)

fig.suptitle(f"Trade-off Latência vs. Throughput — {rf_model.model_name}",
             fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(str(dirs["figures"] / "02_batch_size_tradeoff.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Cross-Validation com Profiling de Tempo

A cross-validation é essencial para avaliar a estabilidade do modelo,
mas tem custo computacional proporcional ao número de folds:

$$T_{cv} = k \cdot (T_{fit} + T_{score})$$

onde k = número de folds.

In [ ]:
# Cross-validation para todos os modelos
# Usar X_train + X_val para CV (sem test)
X_cv = np.vstack([X_train, X_val])
y_cv = np.concatenate([y_train, y_val])

cv_results = []
for name in LatencyBenchmarkModel.SUPPORTED_MODELS:
    print(f"Cross-validating {name}...")
    cv = cross_validate_model(X_cv, y_cv, model_name=name, cv=5, random_state=42)
    cv_results.append(cv)

cv_df = pd.DataFrame(cv_results)

print("\n" + "=" * 80)
print("CROSS-VALIDATION — RESULTADOS")
print("=" * 80)
display_cols = ["model_name", "test_accuracy_mean", "test_accuracy_std",
                "test_f1_mean", "fit_time_mean_s", "total_cv_time_s"]
print(cv_df[display_cols].to_string(index=False))

## 7. Hyperparameter Tuning com Análise de Custo

O tuning de hiperparâmetros é uma das etapas mais custosas do pipeline de ML.
Conforme Li et al. (2014), estratégias eficientes de busca são essenciais
para escalar sistemas de ML.

Vamos medir o custo computacional do GridSearchCV para diferentes algoritmos.

In [ ]:
# Hyperparameter tuning para 2 modelos (tempo razoável)
tuning_results = []

for name in ["logistic_regression", "random_forest"]:
    print(f"Tuning {name}...")
    result = hyperparameter_tuning(
        X_train, y_train, model_name=name, cv=3, random_state=42
    )
    tuning_results.append(result)
    print(f"  Best score: {result['best_score']:.4f}")
    print(f"  Best params: {result['best_params']}")
    print(f"  Time: {result['tuning_time_s']:.2f}s ({result['n_fits']} fits)")
    print()

tuning_df = pd.DataFrame(tuning_results)
print(tuning_df[["model_name", "best_score", "tuning_time_s", "n_candidates", "n_fits"]].to_string(index=False))

## 8. Salvar Modelos e Métricas

Conforme Sculley et al. (2015), a serialização de modelos é parte
do pipeline de ML e impacta a latência de carregamento em produção.

In [ ]:
# Salvar todos os modelos treinados
model_dir = dirs["models"]

for model, metrics in results:
    model_path = str(model_dir / f"{model.model_name}.pkl")
    
    # Medir latência de serialização
    start = time.perf_counter()
    save_model(model, model_path)
    save_time = (time.perf_counter() - start) * 1000
    
    print(f"Modelo '{model.model_name}' salvo em {save_time:.2f} ms")

# Salvar métricas de treinamento
save_metrics(
    {"training": training_metrics, "benchmarks": benchmark_results},
    str(dirs["logs"] / "training_metrics.json"),
)
print("\nMétricas salvas em outputs/logs/training_metrics.json")

## 📋 Resumo

### O que aprendemos neste notebook:

1. **Latência de treinamento**: Logistic Regression é o mais rápido; SVM e GB são mais lentos
2. **Latência de inferência**: Logistic Regression tem vantagem clara; RF e GB escalam com nº de árvores
3. **Percentis de latência**: P99 pode ser significativamente maior que P50 (tail latency)
4. **Batch size**: Aumentar o batch melhora throughput mas pode piorar latência individual
5. **Custo de tuning**: GridSearch é custoso — número de fits = candidatos × folds

### 🔗 Próximo Notebook

No **Notebook 03 — Avaliação**, vamos:
- Avaliar qualidade dos modelos (accuracy, F1, ROC-AUC)
- Analisar o trade-off entre qualidade e latência
- Gerar relatório final comparativo

### 📚 Referências

- Reddi, V. J. et al. (2020). *MLPerf Inference Benchmark*. ISCA '20.
- Dean, J. & Barroso, L. A. (2013). *The Tail at Scale*. CACM.
- Li, M. et al. (2014). *Scaling Distributed Machine Learning*. OSDI '14.